# MCP — Chapter 2: MCP Protocol

This chapter covers how an MCP client and server communicate.

### Topics
- JSON-RPC
- Requests and responses
- Initialization
- Capability negotiation
- Tool discovery
- Tool calls
- MCP lifecycle
- `stdio` and Streamable HTTP

## 1. JSON-RPC

MCP uses **JSON-RPC** as its message format.

Example request:

```json
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "add",
  "params": {
    "a": 10,
    "b": 20
  }
}
```

Example response:

```json
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": 30
}
```

Conceptually:

```text
Client
  │
  │ Call add(10, 20)
  ▼
Server
  │
  │ Result = 30
  ▼
Client
```

JSON-RPC gives MCP a **structured and predictable communication format**.

## 2. JSON-RPC Requests

A request generally contains:

```text
jsonrpc
id
method
params
```

Example:

```json
{
  "jsonrpc": "2.0",
  "id": 42,
  "method": "tools/call",
  "params": {
    "name": "add",
    "arguments": {
      "a": 10,
      "b": 20
    }
  }
}
```

| Field | Meaning |
|---|---|
| `jsonrpc` | JSON-RPC version |
| `id` | Identifies the request |
| `method` | What the client wants |
| `params` | Arguments for that operation |

`"method": "tools/call"` means the client wants to call a tool.

## 3. JSON-RPC Responses

The server responds using the same `id`.

```json
{
  "jsonrpc": "2.0",
  "id": 42,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "30"
      }
    ]
  }
}
```

The `id` lets the client match a response to its request.

For example:

```text
Request 41 → get_user()
Request 42 → calculate()
Request 43 → search()

id 41 → get_user result
id 42 → calculate result
id 43 → search result
```

## 4. Initialization

Before normal communication begins, the client and server establish the connection.

```text
Client
  │
  │ initialize
  ▼
Server
  │
  │ initialization response
  ▼
Client
```

The client provides information about itself and what it supports.

The server responds with its information and capabilities.

This is part of **initialization and capability negotiation**.

## 5. Capability Negotiation

Different MCP implementations can support different features.

For example:

```text
Tools
Resources
Prompts
```

During initialization, the client and server exchange information about their capabilities.

```text
Client
 │
 │ "I support X"
 ▼
Server
 │
 │ "I support Tools + Resources"
 ▼
Client
```

This lets both sides understand what functionality is available.

## 6. Tool Discovery

After initialization, the client can discover the tools provided by the server.

```text
Client
   │
   │ tools/list
   ▼
Server
   │
   ├── calculate
   ├── search_database
   └── get_user
   │
   ▼
Client
```

The server also provides tool descriptions and input schemas.

Example:

```json
{
  "name": "add",
  "description": "Adds two numbers",
  "inputSchema": {
    "type": "object",
    "properties": {
      "a": {"type": "number"},
      "b": {"type": "number"}
    },
    "required": ["a", "b"]
  }
}
```

The client now knows what the tool does and what inputs it accepts.

## 7. Calling a Tool

Suppose the user asks:

> "What is 10 + 20?"

The agent decides to use:

```text
add(a=10, b=20)
```

Flow:

```text
User
 │
 ▼
LLM / Agent
 │
 ▼
MCP Client
 │
 │ tools/call
 ▼
MCP Server
 │
 │ executes tool
 ▼
Result
 │
 ▼
MCP Client
 │
 ▼
Agent
 │
 ▼
User
```

A conceptual request:

```json
{
  "jsonrpc": "2.0",
  "id": 7,
  "method": "tools/call",
  "params": {
    "name": "add",
    "arguments": {
      "a": 10,
      "b": 20
    }
  }
}
```

## 8. Complete MCP Lifecycle

```text
          CLIENT                         SERVER

             │
             │──── initialize ──────────►│
             │                            │
             │◄── initialization ────────│
             │                            │
             │──── initialized ──────────►│
             │                            │
             │──── tools/list ──────────►│
             │                            │
             │◄── available tools ───────│
             │                            │
             │──── tools/call ──────────►│
             │                            │
             │◄──── tool result ──────────│
             │                            │
```

### Mental model

**Initialize → Discover → Call → Receive result**

## 9. Transport: `stdio`

With **`stdio`**, the client launches the MCP server as a local process and communicates through standard input/output.

```text
Client
 │
 │ stdin / stdout
 ▼
MCP Server
```

This is commonly used for local MCP servers.

## 10. Transport: Streamable HTTP

With **Streamable HTTP**, the client communicates with an MCP server over HTTP.

```text
Client
 │
 │ HTTP
 ▼
MCP Server
```

This is useful when the server runs separately from the client.

| `stdio` | Streamable HTTP |
|---|---|
| Usually local | Can be remote |
| Client launches server process | Server can run independently |
| stdin/stdout | HTTP |
| Useful for local integrations | Useful for deployed services |

# 🧠 Chapter 2 — Quick Check

Answer these in your own words.



1. What is JSON-RPC and why does MCP use it?



2. Why does a JSON-RPC request have an `id`?



3. What happens during MCP initialization?



4. What is **capability negotiation**?



5. What does `tools/list` do?



6. What is the difference between `tools/list` and `tools/call`?



7. What's the difference between `stdio` and Streamable HTTP?



### Next Chapter

**Chapter 3 — MCP Primitives**

We'll start coding here and work with:

```text
Tools
Resources
Prompts
```

We'll build our **first MCP server** using Python.